### Instruksi Pengerjaan

Buat notebook baru **`Tugas4_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` baru, lalu kerjakan bagian **A sampai E** berikut — **seluruhnya wajib menggunakan PySpark, bukan pandas**, dan data **wajib dibaca langsung dari HDFS** (`hdfs://localhost:9000/...`), bukan dari berkas lokal.

**SprakSession**

In [2]:
# Sel 1: Inisialisasi PySpark (Code Cell)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum, count, avg, round, isnan

# Membuat SparkSession dengan nama yang representatif
spark = SparkSession.builder \
    .appName("Tugas4_BigData") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat dan siap digunakan.")

26/09/10 04:10:24 WARN Utils: Your hostname, Ubuntu24 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/10 04:10:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 04:10:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat dan siap digunakan.


**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).

In [2]:
# Sel 2: Bagian A (Code Cell)
# Membaca data langsung dari HDFS sesuai instruksi
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df_tugas = spark.read.csv(path_hdfs, header=True, inferSchema=True)

print("=== Skema Data ===")
df_tugas.printSchema()

print(f"=== Jumlah Baris: {df_tugas.count()} ===")

print("=== 10 Baris Pertama ===")
# Menggunakan truncate=False agar teks tidak terpotong (nilai plus untuk kerapian)
df_tugas.show(10, truncate=False)

=== Skema Data ===
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

=== Jumlah Baris: 1000 ===
=== 10 Baris Pertama ===
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |
|ORD-300

**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [3]:
# Sel 3: Bagian B - Menghitung Data Kosong (Code Cell)
# Menghitung jumlah nilai null atau NaN pada kolom rating
jumlah_kosong = df_tugas.filter(col("rating").isNull() | isnan("rating")).count()
print(f"Jumlah baris dengan rating kosong: {jumlah_kosong}")

# Kita memilih untuk menghapus (drop) baris yang tidak memiliki rating
df_bersih = df_tugas.na.drop(subset=["rating"])
print(f"Jumlah baris setelah data kosong ditangani: {df_bersih.count()}")

Jumlah baris dengan rating kosong: 204
Jumlah baris setelah data kosong ditangani: 796


Alasan Penanganan Data Kosong:

Saya memilih menggunakan fungsi df.na.drop() untuk menghapus baris yang nilai rating-nya kosong. Dalam konteks ulasan e-commerce, jika seorang pembeli tidak memberikan rating sama sekali, kita tidak bisa mengasumsikannya sebagai nilai 0 (karena 0 akan sangat menjatuhkan rata-rata seolah-olah barangnya sangat buruk). Kita juga tidak ideal jika mengisinya menggunakan df.na.fill() dengan nilai rata-rata (mean), karena hal tersebut berisiko mengaburkan sentimen asli pelanggan. Oleh karena itu, membuang data yang kosong adalah langkah paling aman agar metrik rata-rata rating tetap representatif.

**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya 

In [4]:
# Sel 5: Bagian C (Code Cell)
# Menambahkan total pendapatan
df_bersih = df_bersih.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Menambahkan tier transaksi menggunakan logika when() dan otherwise()
df_bersih = df_bersih.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

# Memverifikasi hasil transformasi
df_bersih.select("order_id", "total_pendapatan", "tier_transaksi").show(5)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
+--------+----------------+--------------+
only showing top 5 rows



**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

In [5]:
# Sel 6: Bagian D (Code Cell)
print("1. Kategori dengan total_pendapatan tertinggi:")
df_bersih.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan_kategori")) \
    .orderBy(col("total_pendapatan_kategori").desc()) \
    .show(1) # Hanya menampilkan 1 teratas

print("2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:")
df_bersih.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1)

print("3. Rata-rata rating untuk masing-masing metode_pembayaran:")
# Menggunakan fungsi round() agar hasil rata-rata lebih rapi (2 angka di belakang koma)
df_bersih.groupBy("metode_pembayaran") \
    .agg(round(avg("rating"), 2).alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show()

1. Kategori dengan total_pendapatan tertinggi:
+------------+-------------------------+
|    kategori|total_pendapatan_kategori|
+------------+-------------------------+
|Rumah Tangga|                108285000|
+------------+-------------------------+
only showing top 1 row

2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    74|
+----+----------------------+
only showing top 1 row

3. Rata-rata rating untuk masing-masing metode_pembayaran:
+-----------------+----------------+
|metode_pembayaran|rata_rata_rating|
+-----------------+----------------+
|              COD|            4.17|
|    Transfer Bank|            4.16|
|         E-Wallet|            4.14|
|     Kartu Kredit|            4.11|
+-----------------+----------------+



**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

In [7]:
# Sel 7: Bagian E - Menyimpan ke HDFS (Code Cell)
path_simpan_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan_september"

# Menyimpan data dalam format CSV. mode("overwrite") digunakan agar tidak error jika dijalankan ulang
df_bersih.write.csv(path_simpan_hdfs, header=True, mode="overwrite")
print("Data berhasil disimpan ke HDFS.")

[Stage 22:>                                                         (0 + 1) / 1]

Data berhasil disimpan ke HDFS.


In [8]:
# Sel 8: Memverifikasi penyimpanan di HDFS (Code Cell)
# Menjalankan perintah terminal di Jupyter menggunakan tanda seru (!)
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_olahan_september

Found 2 items
-rw-r--r--   3 nexvandar supergroup          0 2026-09-10 03:33 /user/mahasiswa/tugas4/hasil_olahan_september/_SUCCESS
-rw-r--r--   3 nexvandar supergroup      73357 2026-09-10 03:33 /user/mahasiswa/tugas4/hasil_olahan_september/part-00000-4384de93-be3a-4e50-9995-386549d198d2-c000.csv


> **Catatan:** Spark menyimpan hasil sebagai **beberapa berkas partisi** (`part-00000...`, dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark. Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi

**Penjelasan Mengapa Spark Menyimpan Hasil Sebagai Beberapa Berkas Partisi:**

Spark secara bawaan tidak menyimpan hasil akhir ke dalam satu fail tunggal seperti pandas karena Spark dibangun di atas arsitektur pemrosesan terdistribusi (RRD/DataFrame). Saat bekerja, data dipecah (dipartisi) dan diproses secara paralel oleh beberapa executor (atau inti CPU) secara bersamaan. Saat kita menginstruksikan Spark untuk menyimpan data ke 'disk', Spark langsung menyuruh setiap executor untuk menulis bagian memorinya masing-masing ke dalam fail partisi yang terpisah (misalnya part-00000..., part-00001...). Hal ini dilakukan agar proses penulisan (I/O throughput) jauh lebih cepat dan terhindar dari bottleneck, alih-alih memaksa seluruh executor mengantre untuk menulis ke dalam satu fail yang sama.

In [3]:
# Sel 10: Menutup sesi (Code Cell)
# Praktik yang baik di akhir notebook
spark.stop()

**F. Explorasi Tambahan**

**Eksplorasi 1: Analisis Tren Harian (Manipulasi Tipe Data Tanggal)**

Secara bawaan, kolom tanggal masih berupa teks (string). Kita bisa mengubahnya menjadi tipe data Date sungguhan, lalu mengekstrak nama hari untuk melihat di hari apa transaksi paling ramai terjadi.

In [9]:
from pyspark.sql.functions import to_date, date_format

print("--- Eksplorasi 1: Hari dengan Pendapatan Tertinggi ---")
# 1. Mengubah kolom 'tanggal' dari String menjadi DateType
df_eksplorasi = df_bersih.withColumn("tanggal_date", to_date(col("tanggal"), "yyyy-MM-dd"))

# 2. Mengekstrak nama hari (misal: Monday, Tuesday) dari tanggal tersebut
df_eksplorasi = df_eksplorasi.withColumn("nama_hari", date_format(col("tanggal_date"), "EEEE"))

# 3. Agregasi untuk melihat total pendapatan dan jumlah transaksi per hari
tren_harian = df_eksplorasi.groupBy("nama_hari") \
    .agg(
        count("order_id").alias("jumlah_order"),
        spark_sum("total_pendapatan").alias("total_pendapatan_harian")
    ) \
    .orderBy(col("total_pendapatan_harian").desc())

tren_harian.show()

--- Eksplorasi 1: Hari dengan Pendapatan Tertinggi ---


[Stage 23:==========================================================(1 + 0) / 1]

+---------+------------+-----------------------+
|nama_hari|jumlah_order|total_pendapatan_harian|
+---------+------------+-----------------------+
|  Tuesday|         120|               93220000|
|   Monday|         117|               90665000|
|Wednesday|         135|               89685000|
| Thursday|         120|               88615000|
|   Sunday|         113|               83300000|
| Saturday|         104|               79785000|
|   Friday|          87|               72420000|
+---------+------------+-----------------------+



**Eksplorasi 2: Preferensi Pembayaran per Kota (Pivot Table)**

Di Bagian D, kita hanya melihat total transaksi per metode pembayaran. Bagaimana jika kita ingin melihat perbandingannya di setiap kota secara silang? Kita bisa menggunakan fungsi .pivot() yang sangat kuat di PySpark.

In [10]:
print("--- Eksplorasi 2: Pivot Table Metode Pembayaran vs Kota ---")
# Membuat matriks silang (cross-tabulation) antara Kota dan Metode Pembayaran
# df.na.fill(0) digunakan untuk mengisi nilai null (jika ada kombinasi kota & pembayaran yang kosong) dengan angka 0
tabel_pivot = df_bersih.groupBy("kota") \
    .pivot("metode_pembayaran") \
    .count() \
    .na.fill(0)

tabel_pivot.show()

--- Eksplorasi 2: Pivot Table Metode Pembayaran vs Kota ---
+----------+---+--------+------------+-------------+
|      kota|COD|E-Wallet|Kartu Kredit|Transfer Bank|
+----------+---+--------+------------+-------------+
|  Magelang| 31|      31|          32|           23|
|  Semarang| 25|      34|          27|           38|
|   Kebumen| 42|      25|          43|           25|
|      Solo| 41|      44|          30|           40|
| Purworejo| 26|      33|          30|           34|
|Yogyakarta| 38|      32|          29|           43|
+----------+---+--------+------------+-------------+



**Eksplorasi 3: Mencari Transaksi "Paling Sultan" di Tiap Kategori (Window Functions)**

Bagaimana jika kita ingin mencari 1 transaksi dengan total_pendapatan paling tinggi di masing-masing kategori, namun kita tidak ingin merangkumnya (kita ingin tetap melihat order_id dan kota-nya)? Kita tidak bisa pakai sekadar groupBy. Kita harus menggunakan Window Function.

In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

print("--- Eksplorasi 3: Transaksi Tertinggi di Masing-masing Kategori (Window Function) ---")

# 1. Mendefinisikan 'Jendela' (Window): Kelompokkan per kategori, urutkan pendapatan dari tertinggi ke terendah
window_spec = Window.partitionBy("kategori").orderBy(col("total_pendapatan").desc())

# 2. Memberikan ranking pada tiap baris di dalam jendelanya masing-masing
df_ranked = df_bersih.withColumn("ranking", rank().over(window_spec))

# 3. Filter hanya mengambil ranking 1 (transaksi tertinggi di kategorinya), lalu buang kolom ranking
top_transaksi_per_kategori = df_ranked.filter(col("ranking") == 1).drop("ranking")

# Menampilkan hasil (memilih beberapa kolom saja agar rapi)
top_transaksi_per_kategori.select("kategori", "order_id", "kota", "total_pendapatan", "metode_pembayaran").show(truncate=False)

--- Eksplorasi 3: Transaksi Tertinggi di Masing-masing Kategori (Window Function) ---
+----------------------+--------+---------+----------------+-----------------+
|kategori              |order_id|kota     |total_pendapatan|metode_pembayaran|
+----------------------+--------+---------+----------------+-----------------+
|Elektronik            |ORD-3512|Semarang |3850000         |E-Wallet         |
|Fashion               |ORD-3797|Solo     |3850000         |Transfer Bank    |
|Kesehatan & Kecantikan|ORD-3366|Purworejo|3850000         |Kartu Kredit     |
|Kesehatan & Kecantikan|ORD-3414|Purworejo|3850000         |E-Wallet         |
|Makanan & Minuman     |ORD-3477|Magelang |3850000         |E-Wallet         |
|Makanan & Minuman     |ORD-3931|Purworejo|3850000         |Kartu Kredit     |
|Olahraga              |ORD-3261|Solo     |3850000         |Transfer Bank    |
|Olahraga              |ORD-3743|Purworejo|3850000         |E-Wallet         |
|Rumah Tangga          |ORD-3385|Purworejo|38